# Monte Carlo Dropout Metrics

## 1. Imports + configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/msc_thesis/notebooks')

In [3]:
!pip install segmentation_models_pytorch

In [4]:
from pathlib import Path
from dataclasses import dataclass
from typing import Sequence, Optional, List, Tuple, Dict
import random

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# ----------------------------
# EDIT ONLY THIS CONFIG BLOCK
# ----------------------------
IMG_PATH = Path("../inputs/map/map.tif")
LBL_PATH = Path("../inputs/label/para_three_class.tif")
OUTPUT_DIR = Path("../outputs/unified_unet_mcdo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 3
IN_CHANNELS = 6
NODATA_VALUE = -9999
IGNORE_INDEX = -9999

PATCH_SIZE = 256
BLOCK_SIZE = 2048
SPLIT_RATIOS = (0.70, 0.15, 0.15)

SEED = 42
T_MC = 50
DROP_P = 0.20
EPS = 1e-12

# Training settings inherited from the thesis workflow.
LR = 0.000116
WEIGHT_DECAY = 0.000478
BATCH_SIZE = 4
CLASS_WEIGHT_BETA = 0.205623
AUG_PROB = 0.230711
NOISE_STD = 0.074686
MAX_EPOCHS = 50
PATIENCE = 8

# Set False if a compatible model_best.pt already exists.
RUN_TRAINING = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CKPT_PATH = OUTPUT_DIR / "model_best.pt"
TRAIN_CSV = OUTPUT_DIR / "train_patches.csv"
VAL_CSV = OUTPUT_DIR / "val_patches.csv"
TEST_CSV = OUTPUT_DIR / "test_patches.csv"
MEAN_PATH = OUTPUT_DIR / "train_mean.npy"
STD_PATH = OUTPUT_DIR / "train_std.npy"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
RASTER_DIR = OUTPUT_DIR / "metric_rasters"
RASTER_DIR.mkdir(parents=True, exist_ok=True)

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)

# Stage large input rasters on Colab's local disk to avoid repeated Drive I/O.
# The original Drive paths remain unchanged for all persistent outputs/checkpoints.
STAGE_DRIVE_INPUTS = True
STAGE_LOCAL_METRIC_OUTPUTS = True
DRIVE_RASTER_DIR = RASTER_DIR

if STAGE_DRIVE_INPUTS and str(IMG_PATH.resolve()).startswith("/content/drive/"):
    import shutil

    local_input_dir = Path("/content/thesis_inputs")

    def stage_raster(source: Path, subdir: str) -> Path:
        target = local_input_dir / subdir / source.name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists() or target.stat().st_size != source.stat().st_size:
            print(f"Copying {source} to {target}...")
            shutil.copy2(source, target)
        return target

    IMG_PATH = stage_raster(IMG_PATH, "map")
    LBL_PATH = stage_raster(LBL_PATH, "label")

# Write 11 GeoTIFFs locally during inference, then copy them to Google Drive.
# This avoids thousands of small Drive writes and keeps outputs persistent.
if STAGE_LOCAL_METRIC_OUTPUTS and str(DRIVE_RASTER_DIR.resolve()).startswith("/content/drive/"):
    RASTER_DIR = Path("/content/thesis_metric_rasters")
    RASTER_DIR.mkdir(parents=True, exist_ok=True)

assert IMG_PATH.exists(), f"Missing image raster: {IMG_PATH}"
assert LBL_PATH.exists(), f"Missing label raster: {LBL_PATH}"

print("Device:", DEVICE)
print("Image:", IMG_PATH)
print("Label:", LBL_PATH)
print("Outputs:", OUTPUT_DIR)

Copying ../inputs/map/map.tif to /content/thesis_inputs/map/map.tif...
Copying ../inputs/label/para_three_class.tif to /content/thesis_inputs/label/para_three_class.tif...
Device: cuda
Image: /content/thesis_inputs/map/map.tif
Label: /content/thesis_inputs/label/para_three_class.tif
Outputs: ../outputs/unified_unet_mcdo


## 2. Patch utilities, spatial split, normalization, dataset

In [5]:
@dataclass(frozen=True)
class PatchSpec:
    row_off: int
    col_off: int
    height: int
    width: int


def build_training_patch_grid(raster_path: Path, patch_size: int) -> List[PatchSpec]:
    # Training patches are full-size and non-overlapping.
    with rasterio.open(raster_path) as src:
        H, W = src.height, src.width

    return [
        PatchSpec(r, c, patch_size, patch_size)
        for r in range(0, H - patch_size + 1, patch_size)
        for c in range(0, W - patch_size + 1, patch_size)
    ]


def build_full_extent_grid(raster_path: Path, patch_size: int) -> List[PatchSpec]:
    # Inference includes smaller edge windows; dataset pads them before prediction.
    with rasterio.open(raster_path) as src:
        H, W = src.height, src.width

    patches = []
    for r in range(0, H, patch_size):
        h = min(patch_size, H - r)
        for c in range(0, W, patch_size):
            w = min(patch_size, W - c)
            patches.append(PatchSpec(r, c, h, w))
    return patches


def has_valid_label(lbl_path: Path, p: PatchSpec) -> bool:
    with rasterio.open(lbl_path) as src:
        y = src.read(1, window=Window(p.col_off, p.row_off, p.width, p.height))
    return np.any((y != IGNORE_INDEX) & (y >= 0) & (y < NUM_CLASSES))


def spatial_block_split(patches, block_size, ratios=(0.70, 0.15, 0.15), seed=42):
    blocks: Dict[Tuple[int, int], List[PatchSpec]] = {}
    for p in patches:
        key = (p.row_off // block_size, p.col_off // block_size)
        blocks.setdefault(key, []).append(p)

    ids = list(blocks)
    rng = random.Random(seed)
    rng.shuffle(ids)

    n_train = int(ratios[0] * len(ids))
    n_val = int(ratios[1] * len(ids))

    train_ids = set(ids[:n_train])
    val_ids = set(ids[n_train:n_train+n_val])
    test_ids = set(ids[n_train+n_val:])

    train = [p for k in train_ids for p in blocks[k]]
    val = [p for k in val_ids for p in blocks[k]]
    test = [p for k in test_ids for p in blocks[k]]
    return train, val, test


def save_patches(patches, path):
    pd.DataFrame([p.__dict__ for p in patches]).to_csv(path, index=False)


def load_patches(path):
    df = pd.read_csv(path)
    return [PatchSpec(int(r.row_off), int(r.col_off), int(r.height), int(r.width))
            for r in df.itertuples(index=False)]


def estimate_mean_std(img_path, patches, max_patches=400, seed=42):
    rng = np.random.default_rng(seed)
    sample = rng.choice(np.array(patches, dtype=object),
                        size=min(max_patches, len(patches)), replace=False)

    sums = np.zeros(IN_CHANNELS, dtype=np.float64)
    sumsqs = np.zeros(IN_CHANNELS, dtype=np.float64)
    counts = np.zeros(IN_CHANNELS, dtype=np.int64)

    with rasterio.open(img_path) as src:
        for p in sample:
            x = src.read(window=Window(p.col_off, p.row_off, p.width, p.height)).astype(np.float32)
            for c in range(IN_CHANNELS):
                vals = x[c][x[c] != NODATA_VALUE].astype(np.float64)
                if vals.size:
                    sums[c] += vals.sum()
                    sumsqs[c] += np.square(vals).sum()
                    counts[c] += vals.size

    mean = sums / np.maximum(counts, 1)
    var = sumsqs / np.maximum(counts, 1) - mean**2
    return mean.astype(np.float32), np.sqrt(np.maximum(var, 1e-12)).astype(np.float32)


class RasterPatchDataset(Dataset):
    def __init__(self, img_path, lbl_path, patches, mean, std,
                 patch_size=256, augment=False, aug_prob=0.0, noise_std=0.0, seed=42):
        self.img_path = str(img_path)
        self.lbl_path = str(lbl_path)
        self.patches = list(patches)
        self.mean = mean.reshape(-1, 1, 1)
        self.std = std.reshape(-1, 1, 1)
        self.patch_size = patch_size
        self.augment = augment
        self.aug_prob = aug_prob
        self.noise_std = noise_std
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.patches)

    def _augment(self, x, y):
        if self.rng.random() > self.aug_prob:
            return x, y
        if self.rng.random() < 0.5:
            x, y = x[:, :, ::-1].copy(), y[:, ::-1].copy()
        if self.rng.random() < 0.5:
            x, y = x[:, ::-1, :].copy(), y[::-1, :].copy()
        k = int(self.rng.integers(0, 4))
        if k:
            x = np.rot90(x, k, axes=(1, 2)).copy()
            y = np.rot90(y, k, axes=(0, 1)).copy()
        if self.noise_std > 0:
            valid = x != NODATA_VALUE
            noise = self.rng.normal(0, self.noise_std, x.shape).astype(np.float32)
            x = np.where(valid, x + noise, x)
        return x, y

    def __getitem__(self, idx):
        p = self.patches[idx]
        win = Window(p.col_off, p.row_off, p.width, p.height)

        with rasterio.open(self.img_path) as img, rasterio.open(self.lbl_path) as lbl:
            x = img.read(window=win).astype(np.float32)
            y = lbl.read(1, window=win).astype(np.int64)

        if self.augment:
            x, y = self._augment(x, y)

        valid_x = x != NODATA_VALUE
        x = np.where(valid_x, (x - self.mean) / self.std, 0.0).astype(np.float32)

        # Pad edge patches during full-area inference.
        _, h, w = x.shape
        if h != self.patch_size or w != self.patch_size:
            xp = np.zeros((IN_CHANNELS, self.patch_size, self.patch_size), np.float32)
            yp = np.full((self.patch_size, self.patch_size), IGNORE_INDEX, np.int64)
            xp[:, :h, :w] = x
            yp[:h, :w] = y
            x, y = xp, yp

        return torch.from_numpy(x), torch.from_numpy(y), idx


# Build once; reuse existing split/normalization files when available.
if all(p.exists() for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, MEAN_PATH, STD_PATH]):
    train_patches = load_patches(TRAIN_CSV)
    val_patches = load_patches(VAL_CSV)
    test_patches = load_patches(TEST_CSV)
    train_mean, train_std = np.load(MEAN_PATH), np.load(STD_PATH)
    print("Loaded existing splits and normalization.")
else:
    patches = [p for p in build_training_patch_grid(LBL_PATH, PATCH_SIZE) if has_valid_label(LBL_PATH, p)]
    train_patches, val_patches, test_patches = spatial_block_split(
        patches, BLOCK_SIZE, SPLIT_RATIOS, SEED
    )
    save_patches(train_patches, TRAIN_CSV)
    save_patches(val_patches, VAL_CSV)
    save_patches(test_patches, TEST_CSV)

    train_mean, train_std = estimate_mean_std(IMG_PATH, train_patches, seed=SEED)
    np.save(MEAN_PATH, train_mean)
    np.save(STD_PATH, train_std)
    print("Created splits and training-only normalization.")

print("Train / val / test:", len(train_patches), len(val_patches), len(test_patches))
print("Mean:", train_mean)
print("Std :", train_std)

Loaded existing splits and normalization.
Train / val / test: 1936 408 512
Mean: [ 8356.905   8785.753   8015.2617 18182.428  12833.63    9659.891 ]
Std : [ 897.9387   677.70233  516.11804 2065.2412  2160.6301  1493.2701 ]


## 3. Explicit-dropout ResNet34 U-Net

In [6]:
class ConvBnReluDrop(nn.Module):
    def __init__(self, in_ch, out_ch, p=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=p),
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, p=0.2):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv1 = ConvBnReluDrop(in_ch + skip_ch, out_ch, p)
        self.conv2 = ConvBnReluDrop(out_ch, out_ch, p)

    def forward(self, x, skip=None):
        x = self.up(x)

        if skip is not None:
            # Ensure encoder and decoder spatial dimensions match.
            if x.shape[-2:] != skip.shape[-2:]:
                x = nn.functional.interpolate(
                    x,
                    size=skip.shape[-2:],
                    mode="nearest"
                )

            x = torch.cat([x, skip], dim=1)

        return self.conv2(self.conv1(x))


class DropoutDecoder(nn.Module):
    def __init__(
        self,
        encoder_channels,
        decoder_channels=(256, 128, 64, 32, 16),
        p=0.2
    ):
        super().__init__()

        in_ch = encoder_channels[-1]
        skips = list(encoder_channels[-2::-1])

        blocks = []

        for i, out_ch in enumerate(decoder_channels):
            skip_ch = skips[i] if i < len(skips) else 0

            blocks.append(
                DecoderBlock(in_ch, skip_ch, out_ch, p)
            )

            in_ch = out_ch

        self.blocks = nn.ModuleList(blocks)
        self.out_channels = decoder_channels[-1]

    def forward(self, *features):

        # Support both older and newer SMP decoder interfaces.
        if len(features) == 1 and isinstance(
            features[0], (list, tuple)
        ):
            features = features[0]

        x = features[-1]
        skips = list(features[-2::-1])

        for i, block in enumerate(self.blocks):
            skip = skips[i] if i < len(skips) else None
            x = block(x, skip)

        return x


def build_unet_mc_dropout(drop_p=0.2):
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
        activation=None,
    )

    decoder = DropoutDecoder(
        model.encoder.out_channels,
        p=drop_p
    )

    model.decoder = decoder

    model.segmentation_head = nn.Conv2d(
        decoder.out_channels,
        NUM_CLASSES,
        kernel_size=1
    )

    return model


model = build_unet_mc_dropout(DROP_P).to(DEVICE)

dropout_count = sum(
    isinstance(m, nn.Dropout2d)
    for m in model.modules()
)

print("Explicit Dropout2d modules:", dropout_count)

assert dropout_count > 0

# Verify the model produces the expected segmentation dimensions.
model.eval()

with torch.no_grad():
    dummy = torch.zeros(
        1,
        IN_CHANNELS,
        PATCH_SIZE,
        PATCH_SIZE,
        device=DEVICE
    )

    output = model(dummy)

assert output.shape == (
    1,
    NUM_CLASSES,
    PATCH_SIZE,
    PATCH_SIZE
), f"Unexpected output shape: {output.shape}"

print("Model output shape:", tuple(output.shape))


Explicit Dropout2d modules: 10
Model output shape: (1, 3, 256, 256)


## Load the trained checkpoint

In [7]:
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)

model = build_unet_mc_dropout(float(ckpt.get("drop_p", DROP_P))).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

print("Loaded checkpoint:", CKPT_PATH)

Loaded checkpoint: ../outputs/unified_unet_mcdo/model_best.pt


## 5. Monte Carlo Dropout + all 11 metric definitions

In [8]:
def enable_mc_dropout(model):
    # Turn dropout on but freeze BatchNorm statistics.
    model.train()
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
            m.eval()


def entropy_from_probs(probs, eps=1e-12):
    p = probs.clamp(min=eps, max=1.0)
    return -(p * torch.log(p)).sum(dim=1)


@torch.no_grad()
def mc_predict_all_metrics(model, x, y, T=50, eps=1e-12):
    x = x.to(DEVICE)
    y = y.to(DEVICE)
    enable_mc_dropout(model)

    probs_runs = []
    pred_runs = []
    entropy_runs = []

    for _ in range(T):
        probs = torch.softmax(model(x), dim=1)
        probs_runs.append(probs)
        pred_runs.append(probs.argmax(dim=1))
        entropy_runs.append(entropy_from_probs(probs, eps))

    probs_stack = torch.stack(probs_runs)       # [T,B,C,H,W]
    preds_stack = torch.stack(pred_runs)        # [T,B,H,W]
    entropy_stack = torch.stack(entropy_runs)   # [T,B,H,W]

    prob_mean = probs_stack.mean(0)
    pred = prob_mean.argmax(1)
    confidence = prob_mean.max(1).values

    # Thesis uncertainty decomposition.
    aleatoric = (probs_stack * (1.0 - probs_stack)).mean(0).sum(1)
    epistemic = probs_stack.var(0, unbiased=False).sum(1)
    predictive = aleatoric + epistemic

    # Predictive entropy and mutual information.
    entropy = entropy_from_probs(prob_mean, eps)
    expected_entropy = entropy_stack.mean(0)
    mutual_information = entropy - expected_entropy

    # Fraction of MC class predictions differing from final mean prediction.
    disagreement = (preds_stack != pred.unsqueeze(0)).float().mean(0)

    # Reference-dependent metrics.
    valid = (y != IGNORE_INDEX) & (y >= 0) & (y < NUM_CLASSES)
    y_safe = y.clone()
    y_safe[~valid] = 0

    p_true = prob_mean.gather(1, y_safe.unsqueeze(1)).squeeze(1)
    cross_entropy = -torch.log(p_true.clamp(min=eps, max=1.0))
    ce_minus_entropy = cross_entropy - entropy

    one_hot = torch.nn.functional.one_hot(
        y_safe, num_classes=NUM_CLASSES
    ).permute(0, 3, 1, 2).float()
    brier = ((prob_mean - one_hot) ** 2).sum(1)

    correctness = (pred == y).float()
    # This reproduces the pixel-wise calibration-error raster used in qgis_layers.ipynb.
    calibration_error = torch.abs(confidence - correctness)

    return {
        "predictive_uncertainty": predictive,
        "aleatoric_uncertainty": aleatoric,
        "epistemic_uncertainty": epistemic,
        "entropy": entropy,
        "disagreement": disagreement,
        "mutual_information": mutual_information,
        "calibration_error": calibration_error,
        "cross_entropy_loss": cross_entropy,
        "softmax_confidence": confidence,
        "ce_minus_entropy": ce_minus_entropy,
        "brier_score": brier,
        "valid": valid,
    }

## 6. Full-area inference dataset

In [9]:
full_patches = build_full_extent_grid(LBL_PATH, PATCH_SIZE)

# Prefer normalization stored in checkpoint when present.
infer_mean = np.asarray(ckpt.get("train_mean", train_mean), dtype=np.float32)
infer_std = np.asarray(ckpt.get("train_std", train_std), dtype=np.float32)

full_ds = RasterPatchDataset(
    IMG_PATH, LBL_PATH, full_patches,
    infer_mean, infer_std,
    patch_size=PATCH_SIZE,
    augment=False,
)

# Batch size 1 is memory-safe for T=50 and avoids holding many MC stacks at once.
full_loader = DataLoader(full_ds, batch_size=1, shuffle=False, num_workers=0)

print("Full-area patches:", len(full_patches))
print("MC passes per patch:", T_MC)

Full-area patches: 2964
MC passes per patch: 50


## 7. Export all 11 QGIS-ready GeoTIFF layers

In [10]:
METRIC_NAMES = [
    "predictive_uncertainty",
    "aleatoric_uncertainty",
    "epistemic_uncertainty",
    "entropy",
    "disagreement",
    "mutual_information",
    "calibration_error",
    "cross_entropy_loss",
    "softmax_confidence",
    "ce_minus_entropy",
    "brier_score",
]

raster_paths = {name: RASTER_DIR / f"{name}.tif" for name in METRIC_NAMES}

with rasterio.open(LBL_PATH) as ref:
    profile = ref.profile.copy()
    profile.update(
        count=1,
        dtype="float32",
        nodata=np.float32(-9999.0),
        compress="deflate",
        tiled=True,
        blockxsize=256,
        blockysize=256,
        BIGTIFF="IF_SAFER",
    )

# Create output rasters.
for path in raster_paths.values():
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(np.full((dst.height, dst.width), -9999.0, dtype=np.float32), 1)

writers = {name: rasterio.open(path, "r+") for name, path in raster_paths.items()}

try:
    for batch_i, (x_b, y_b, idx_b) in enumerate(full_loader, start=1):
        outputs = mc_predict_all_metrics(model, x_b, y_b, T=T_MC, eps=EPS)
        idx = int(idx_b.item())
        p = full_patches[idx]
        win = Window(p.col_off, p.row_off, p.width, p.height)

        valid = outputs["valid"][0, :p.height, :p.width].cpu().numpy()

        for name in METRIC_NAMES:
            values = outputs[name][0, :p.height, :p.width].detach().cpu().numpy().astype(np.float32)
            out_arr = np.where(valid, values, -9999.0).astype(np.float32)
            writers[name].write(out_arr, 1, window=win)

        if batch_i % 25 == 0 or batch_i == len(full_loader):
            print(f"Processed {batch_i}/{len(full_loader)} patches")

finally:
    for dst in writers.values():
        dst.close()

# Persist completed GeoTIFFs to Google Drive after local inference finishes.
# If inference fails, the local files are retained for inspection, not synced as complete.
if RASTER_DIR != DRIVE_RASTER_DIR:
    import shutil
    DRIVE_RASTER_DIR.mkdir(parents=True, exist_ok=True)
    for name, local_path in raster_paths.items():
        drive_path = DRIVE_RASTER_DIR / local_path.name
        print(f"Saving {name} to Google Drive...")
        shutil.copy2(local_path, drive_path)
    print("All 11 metric rasters copied to:", DRIVE_RASTER_DIR)

print("\nExport complete:")
for name, path in raster_paths.items():
    print(f"{name:24s} -> {DRIVE_RASTER_DIR / path.name if RASTER_DIR != DRIVE_RASTER_DIR else path}")

Processed 25/2964 patches
Processed 50/2964 patches
Processed 75/2964 patches
Processed 100/2964 patches
Processed 125/2964 patches
Processed 150/2964 patches
Processed 175/2964 patches
Processed 200/2964 patches
Processed 225/2964 patches
Processed 250/2964 patches
Processed 275/2964 patches
Processed 300/2964 patches
Processed 325/2964 patches
Processed 350/2964 patches
Processed 375/2964 patches
Processed 400/2964 patches
Processed 425/2964 patches
Processed 450/2964 patches
Processed 475/2964 patches
Processed 500/2964 patches
Processed 525/2964 patches
Processed 550/2964 patches
Processed 575/2964 patches
Processed 600/2964 patches
Processed 625/2964 patches
Processed 650/2964 patches
Processed 675/2964 patches
Processed 700/2964 patches
Processed 725/2964 patches
Processed 750/2964 patches
Processed 775/2964 patches
Processed 800/2964 patches
Processed 825/2964 patches
Processed 850/2964 patches
Processed 875/2964 patches
Processed 900/2964 patches
Processed 925/2964 patches
Proc

## 8. Quick output check

In [11]:
summary = []

for name, path in raster_paths.items():
    with rasterio.open(path) as src:
        a = src.read(1)
        valid = a != src.nodata
        summary.append({
            "metric": name,
            "valid_pixels": int(valid.sum()),
            "min": float(a[valid].min()) if valid.any() else np.nan,
            "mean": float(a[valid].mean()) if valid.any() else np.nan,
            "max": float(a[valid].max()) if valid.any() else np.nan,
            "path": str(path),
        })

pd.DataFrame(summary)

,metric,valid_pixels,min,mean,max,path
0,predictive_uncertainty,192790240,7.140813e-04,0.117397,0.666661,/content/thesis_metric_rasters/predictive_unce...
1,aleatoric_uncertainty,192790240,1.519090e-04,0.109728,0.605617,/content/thesis_metric_rasters/aleatoric_uncer...
2,epistemic_uncertainty,192790240,3.261819e-06,0.007669,0.539923,/content/thesis_metric_rasters/epistemic_uncer...
3,entropy,192790240,3.195383e-03,0.224208,1.098604,/content/thesis_metric_rasters/entropy.tif
4,disagreement,192790240,0.000000e+00,0.016721,0.820000,/content/thesis_metric_rasters/disagreement.tif
5,mutual_information,192790240,8.833518e-04,0.022622,0.878931,/content/thesis_metric_rasters/mutual_informat...
6,calibration_error,192790240,3.572106e-04,0.104628,0.997496,/content/thesis_metric_rasters/calibration_err...
7,cross_entropy_loss,192790240,3.572745e-04,0.172388,7.616020,/content/thesis_metric_rasters/cross_entropy_l...
8,softmax_confidence,192790240,3.352057e-01,0.924692,0.999643,/content/thesis_metric_rasters/softmax_confide...
9,ce_minus_entropy,192790240,-4.630552e-01,-0.051821,7.488901,/content/thesis_metric_rasters/ce_minus_entrop...
